# Building micrograd

Environment check — run this first to confirm the kernel and libraries are wired up correctly.

In [15]:
# Environment check — RUN THIS FIRST.
# If the assert at the bottom fails, the kernel is wrong: click the kernel
# picker (top-right) and choose the .venv/bin/python interpreter, then re-run.
import sys, shutil
import math
import random
import numpy as np
np.random.seed(42)
import matplotlib
import torch
import graphviz
import sklearn

print(f"python    : {sys.version.split()[0]}")
print(f"executable: {sys.executable}")
print(f"venv      : {sys.prefix}")
print(f"numpy     : {np.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"torch     : {torch.__version__}")
print(f"sklearn   : {sklearn.__version__}")
print(f"graphviz  : {graphviz.__version__}")
print(f"dot binary: {shutil.which('dot') or 'NOT FOUND -> run: brew install graphviz'}")

assert ".venv" in sys.executable, (
    f"WRONG KERNEL: running {sys.executable}. "
    "Pick the .venv/bin/python kernel (top-right) and re-run."
)
assert shutil.which("dot"), "graphviz 'dot' binary missing -> run: brew install graphviz"
print("\nenvironment OK - kernel is the project .venv and every library imports + renders")


python    : 3.12.12
executable: /Users/pranav/Workspace/neural-networks/.venv/bin/python
venv      : /Users/pranav/Workspace/neural-networks/.venv
numpy     : 1.26.4
matplotlib: 3.10.9
torch     : 2.2.2
sklearn   : 1.8.0
graphviz  : 0.21
dot binary: /usr/local/bin/dot

environment OK - kernel is the project .venv and every library imports + renders


In [16]:
class Value(): 

    def __init__(self, data, _children = (), _op = '', label = ''): 
        self.data = data 
        self.grad = 0.0 
        self._backward = lambda:None 
        self._prev = set(_children) 
        self._op = _op 
        self.label = label 
    
    def __repr__(self): 
        out = f"Value(data = {self.data})" 
        return out 
    
    def __add__(self, other): 
        other = other if isinstance(other, Value) else Value(other) 
        out = Value(self.data + other.data, (self, other), label = '+') 

        def _backward(): 
            self.grad += 1.0 * out.grad 
            other.grad += 1.0 * out.grad 
        out._backward = _backward 

        return out 
    
    def __radd__(self, other): 
        return self + other 

    def __mul__(self, other): 
        other = other if isinstance(other, Value) else Value(other) 
        out = Value(self.data * other.data, (self, other), label = '*') 

        def _backward(): 
            self.grad += other.data * out.grad 
            other.grad += self.data * out.grad 
        out._backward = _backward 

        return out 
    
    def __rmul__(self, other):
        return self * other 

    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = self + (-1.0 * other)

        return out 

    def __rsub__(self, other):
        return Value(other) + (-1.0 * self)
    
    def __pow__(self, other):
        assert(isinstance(other, (int, float)))
        out = Value(self.data ** other, (self, ), f'^{other}')

        def _backward():
            self.grad += out.grad * (other * (self.data ** (other - 1.0)))
        out._backward = _backward

        return out 
    
    def __rpow__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return pow(other, self)
    
    def __truediv__(self, other):
        return self * (other ** (-1))
    
    def __rtruediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return other / self

    def exp(self):
        out = Value(math.exp(self.data), (self, ), label = 'exp')

        def _backward():
            self.grad += math.exp(self.data) * out.grad 
        out._backward = _backward

        return out 

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out
    
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0

        for node in reversed(topo):
            node._backward()

In [17]:
class Neuron():
    
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1.0, 1.0)) for i in range(nin)]
        self.b = Value(random.uniform(-1.0, 1.0))

    def __call__(self, x):
        # act = w*x + b
        act = 0.0 

        for wi, xi in zip(self.w, x): 
            act += wi * xi 
        act += self.b 

        out = act.tanh() 
        return out 

    def parameters(self):
        return self.w + [self.b]

class Layer():

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for i in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        # fix: always returned a list, so a 1-neuron output layer gave back [Value] and broke `yt - yp` with a float-minus-list TypeError -> unwrap when there's only one neuron
        return outs[0] if len(outs) == 1 else outs 

    def parameters(self):
        out = []
        for neuron in self.neurons:
            for p in neuron.parameters():
                out.append(p)
        return out 

class MLP():

    def __init__(self, nin, nouts):
        sz = [nin] + nouts 
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        out = []
        for layer in self.layers:
            for p in layer.parameters():
                out.append(p)
        return out 

In [18]:
a = [float(x) for x in np.random.uniform(-5, 5, 100)]
b = [float(x) for x in np.random.uniform(-5, 5, 100)]
y = [float(x) for x in np.random.choice([-1, 1], 100).astype(float)]

network = MLP(2, [16, 1])
step = 0.05

for epoch in range(100):
    
    # forward pass:
    # take predictions
    ypred = []
    for (ai, bi) in list(zip(a, b)):
        ypred.append(network((ai, bi)))
    # take loss using MSE
    loss = 0.0
    for yt, yp in zip(y, ypred):
        loss += (yt - yp) ** 2
    loss = loss * (1.0 / len(y))
    
    # back pass:
    # reset gradients
    for p in network.parameters():
        p.grad = 0.0
    # backpropagate 
    loss.backward()

    # update 
    for p in network.parameters():
        p.data -= step * p.grad 

    print(f"Epoch: {epoch  + 1}, Loss: {loss.data}")

Epoch: 1, Loss: 1.5318065931318616
Epoch: 2, Loss: 1.4106691041635284
Epoch: 3, Loss: 1.36152490588735
Epoch: 4, Loss: 1.3175924974612707
Epoch: 5, Loss: 1.2725539706689666
Epoch: 6, Loss: 1.2287461075379582
Epoch: 7, Loss: 1.187582166226917
Epoch: 8, Loss: 1.1496573407330133
Epoch: 9, Loss: 1.1161742979676985
Epoch: 10, Loss: 1.0881941447568153
Epoch: 11, Loss: 1.065680092715837
Epoch: 12, Loss: 1.0477350860496215
Epoch: 13, Loss: 1.0332558863566674
Epoch: 14, Loss: 1.021307257544676
Epoch: 15, Loss: 1.0112087018572045
Epoch: 16, Loss: 1.0024986638804154
Epoch: 17, Loss: 0.9948722769739472
Epoch: 18, Loss: 0.9881261963048108
Epoch: 19, Loss: 0.9821188267254124
Epoch: 20, Loss: 0.9767453193625197
Epoch: 21, Loss: 0.9719234477048478
Epoch: 22, Loss: 0.9675860396733833
Epoch: 23, Loss: 0.9636767047525893
Epoch: 24, Loss: 0.9601470235097882
Epoch: 25, Loss: 0.9569544595272116
Epoch: 26, Loss: 0.9540608192152003
Epoch: 27, Loss: 0.9514312449436464
Epoch: 28, Loss: 0.9490336886232419
Epoch: